In [23]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, make_scorer, roc_auc_score, auc


In [11]:
# Läs in träningsfilen
df = pd.read_csv(r"C:\Users\Eier\OneDrive\Dokument\DataScience\Data_science_projektkurs\archive (1)\diabetes_prediction_dataset.csv")
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df['gender'].unique())
print(df['smoking_history'].unique())
df_encoded = pd.get_dummies(df, columns=['gender', 'smoking_history'], drop_first=False)
print(df_encoded.head())
print(df_encoded.columns.tolist())
numeric_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 
                'HbA1c_level', 'blood_glucose_level', 'diabetes']

df_numeric = df_encoded[numeric_cols]
print(df_numeric.describe().T)
print(df['diabetes'].value_counts())

Y = df_encoded['diabetes']
X = df_encoded.drop(columns=['diabetes'])

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB
None
                 age  hypertension  heart_disease            bmi  \
count  100000.000000  100000.00000  100000.000000  100000.000000   
mean       41.885856       0.07485       0.039420      27.320767   
std        22.516840       0.26315       0.194593       6.636783   
min   

In [12]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

# 15% val, 15% test
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42, stratify=Y_temp
)

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)
print(X_test.shape, Y_test.shape)

(70000, 15) (70000,)
(15000, 15) (15000,)
(15000, 15) (15000,)


In [14]:
# Ladda sparade modeller
hgb = joblib.load('best_hgb_model.joblib')
cb = joblib.load('best_cb_model.joblib')  
rf = joblib.load('best_rf_model.joblib')    

# Skapa en voter
voter = VotingClassifier(estimators=[
    ('hgb', hgb),
    ('cb', cb),
    ('rf', rf)
], voting='soft')  # Soft voting använder sannolikheter för bättre prestanda

# Träna votern
voter.fit(X_train, Y_train)
y_pred= voter.predict(X_val)

# Utvärdera votern och generera en klassificeringsrapport som en dictionary
report = classification_report(Y_val, y_pred, output_dict=True)

# Skriv ut klassificeringspoängen
score=voter.score(X_val, Y_val)
print(f"Voting Classifier Score: {score:.4f}")

# Spara Voting Classifier-modell som en pickle-fil
joblib.dump(voter, 'voting_classifier.pkl')
print('Voting Classifier är sparad som voting_classifier.pkl')

Voting Classifier Score: 0.9729
Voting Classifier är sparad som voting_classifier.pkl


In [18]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# -------------------------------------------------------
# Ladda modellen från joblib
# -------------------------------------------------------
voter  = joblib.load("voting_classifier.pkl")
print("[JOBLIB] Voter-Classifier laddad.")

# -------------------------------------------------------
# Prediktioner
# -------------------------------------------------------
preds = hgb.predict(X_test)
probs = hgb.predict_proba(X_test)[:, 1]

# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
acc = accuracy_score(Y_test, preds)
f1 = f1_score(Y_test, preds)
prec = precision_score(Y_test, preds)
rec = recall_score(Y_test, preds)
auc = roc_auc_score(Y_test, probs)
cm = confusion_matrix(Y_test, preds)

# -------------------------------------------------------
# Utskrifter
# -------------------------------------------------------
print("\n====================")
print("   Voter – METRICS")
print("====================")

print(f"Accuracy:      {acc:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Precision:     {prec:.4f}")
print(f"Recall:        {rec:.4f}")
print(f"ROC-AUC:       {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(Y_test, preds))


[JOBLIB] Voter-Classifier laddad.

   Voter – METRICS
Accuracy:      0.9714
F1-score:      0.8002
Precision:     0.9851
Recall:        0.6737
ROC-AUC:       0.9773

Confusion Matrix:
[[13712    13]
 [  416   859]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     13725
           1       0.99      0.67      0.80      1275

    accuracy                           0.97     15000
   macro avg       0.98      0.84      0.89     15000
weighted avg       0.97      0.97      0.97     15000



In [24]:
# Ladda den sparade modellen
voter = joblib.load('voting_classifier.pkl')  # Ändra filnamnet till den sparade modellen

# Definiera scorer
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro')
}

# Beräkna cross-validation scores
cv_results = cross_validate(voter, X_train, Y_train, cv=5, scoring=scoring)

# Skapa en lista med resultaten
voter_results = []
for fold in range(5):
    voter_results.append({
        'Fold': fold,
        'Accuracy': cv_results['test_accuracy'][fold],
        'Precision': cv_results['test_precision'][fold],
        'Recall': cv_results['test_recall'][fold],
        'F1-score': cv_results['test_f1'][fold]
    })

# Konvertera listan till en DataFrame
voter_results = pd.DataFrame(voter_results)
print(voter_results)

# Spara DataFrame med joblib

   Fold  Accuracy  Precision    Recall  F1-score
0     0  0.972643   0.981472  0.841744  0.897405
1     1  0.971857   0.978749  0.838646  0.894313
2     2  0.970000   0.973600  0.830390  0.886714
3     3  0.974000   0.983374  0.848965  0.903161
4     4  0.971714   0.982655  0.835138  0.892989
